# 02 — Cleaning and SQL Analytics

This notebook runs the repository's tested ingestion/cleaning implementation and then executes the DuckDB business queries. It preserves cancellation rows for returns analysis and flags valid purchases separately.

**Prerequisite:** Complete Notebook 01 and place the official workbook at `data/raw/online_retail_II.xlsx`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "data.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import main as ingest_data, PARQUET_FILE, DB_FILE
from src.run_sql import main as run_analytics

print("Project:", PROJECT_ROOT)

Project: /Users/nithiii/Downloads/data-science-ai-portfolio/01-machine-learning/customer-retention-revenue-intelligence


## 1. Ingest, clean and persist

The pipeline standardizes column names, parses dates and numeric fields, removes exact duplicate rows, flags cancellation invoices and defines a valid purchase as a non-cancellation row with positive quantity and unit price. Rows without Customer ID remain available for transaction-level analysis.

In [2]:
ingest_data()
print("Parquet created:", PARQUET_FILE.exists())
print("DuckDB created:", DB_FILE.exists())

Ingestion complete.
   rows  invoices  identified_customers  cancellation_lines  unidentified_customer_lines
1033036     53628                  5942             19104.0                     235151.0
Parquet: /Users/nithiii/Downloads/data-science-ai-portfolio/01-machine-learning/customer-retention-revenue-intelligence/data/processed/transactions.parquet
DuckDB: /Users/nithiii/Downloads/data-science-ai-portfolio/01-machine-learning/customer-retention-revenue-intelligence/data/processed/retail.duckdb
Parquet created: True
DuckDB created: True


## 2. Inspect cleaned transaction flags

In [3]:
clean = pd.read_parquet(PARQUET_FILE)
print("Cleaned rows:", len(clean))
display(clean.head())
display(clean.groupby(["is_cancellation", "is_valid_purchase"], dropna=False).size().rename("rows").to_frame())
print("Valid purchase rows:", int(clean["is_valid_purchase"].sum()))

Cleaned rows: 1033036


,invoice_date,stock_code,description,country,unit_price,invoice_no,customer_id,quantity,is_cancellation,line_revenue,is_valid_purchase
0,2009-12-01 07:45:00,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,United Kingdom,6.95,489434,13085,12,False,83.4,True
1,2009-12-01 07:45:00,79323P,PINK CHERRY LIGHTS,United Kingdom,6.75,489434,13085,12,False,81.0,True
2,2009-12-01 07:45:00,79323W,WHITE CHERRY LIGHTS,United Kingdom,6.75,489434,13085,12,False,81.0,True
3,2009-12-01 07:45:00,22041,"RECORD FRAME 7"" SINGLE SIZE",United Kingdom,2.10,489434,13085,48,False,100.8,True
4,2009-12-01 07:45:00,21232,STRAWBERRY CERAMIC TRINKET BOX,United Kingdom,1.25,489434,13085,24,False,30.0,True


rows
is_cancellation is_valid_purchase         
False           False                 6019
                True               1007913
True            False                19104

Valid purchase rows: 1007913


## 3. Run business SQL

The SQL script exports monthly revenue, country performance, cancellations, RFM segments, cohort retention and customer revenue concentration. These are descriptive analytics, not model performance results.

In [4]:
run_analytics()
SQL_OUTPUT = PROJECT_ROOT / "reports" / "sql"
print("Exported files:", sorted(p.name for p in SQL_OUTPUT.glob("*.csv")))

monthly_revenue: 25 rows
country_revenue: 43 rows
cancellations: 25 rows
rfm_segments: 6 rows
cohort_retention: 325 rows
revenue_concentration: 5,878 rows
Exported files: ['cancellations.csv', 'cohort_retention.csv', 'country_revenue.csv', 'monthly_revenue.csv', 'revenue_concentration.csv', 'rfm_segments.csv']


## 4. Review the outputs

In [5]:
for filename in ["monthly_revenue.csv", "country_revenue.csv", "cancellations.csv", "rfm_segments.csv"]:
    path = SQL_OUTPUT / filename
    if path.exists():
        print(f"\\n{filename}")
        display(pd.read_csv(path).head(10))

\nmonthly_revenue.csv


,month,revenue_gbp,invoices,identified_customers
0,2009-12-01,822483.950,1682,955
1,2010-01-01,651155.112,1105,720
2,2010-02-01,551504.726,1201,772
3,2010-03-01,830915.261,1681,1057
4,2010-04-01,678875.252,1462,942
5,2010-05-01,657705.500,1500,966
6,2010-06-01,749537.310,1645,1041
7,2010-07-01,648810.270,1529,928
8,2010-08-01,695251.910,1425,911
9,2010-09-01,921696.991,1839,1145


\ncountry_revenue.csv


,country,revenue_gbp,invoices,customers
0,United Kingdom,1.741020e+07,36535,5350
1,EIRE,6.587673e+05,626,5
2,Netherlands,5.540381e+05,228,22
3,Germany,4.250197e+05,789,107
4,France,3.504561e+05,622,95
5,Australia,1.692835e+05,95,15
6,Spain,1.083325e+05,154,41
7,Switzerland,1.006856e+05,93,22
8,Sweden,9.186982e+04,105,19
9,Denmark,6.858069e+04,43,12


\ncancellations.csv


,month,cancellation_lines,cancellation_value_gbp
0,2009-12-01,1013,25835.45
1,2010-01-01,686,28675.61
2,2010-02-01,575,20986.50
3,2010-03-01,841,67668.02
4,2010-04-01,626,37354.20
5,2010-05-01,981,44434.78
6,2010-06-01,790,72463.44
7,2010-07-01,734,31444.79
8,2010-08-01,571,40477.52
9,2010-09-01,807,70591.03


\nrfm_segments.csv


,segment,customers,avg_customer_revenue_gbp
0,Hibernating,1640,440.198873
1,Champions,1330,8967.549974
2,Potential loyalists,1214,994.938487
3,Loyal high value,956,3017.386285
4,At risk,433,1172.596282
5,Recent customers,305,412.543770


## Interpretation prompts

- Is revenue concentrated in a small number of countries or customers?
- Do cancellation values vary over time?
- Which RFM segments have the largest customer counts and average revenue?
- How does cohort retention change as month number increases?

Do not infer campaign impact from these descriptive patterns.